# Matched null persistence landscapes: CEBRA versus PCA

This notebook rebuilds the comparison from cleaned neural recordings so that the real and null point clouds are directly matched.

For each region, stimulus, recording, and requested embedding dimension it:

1. standardizes neurons using statistics from the real recording;
2. fits CEBRA and PCA once to the real recording;
3. creates 100 null realizations by independently circularly shifting each neuron;
4. transforms each null through the **same fitted representation** as the real data;
5. uses the same sampled time indices for real/null and CEBRA/PCA point clouds;
6. centers each point cloud and divides it by its RMS radius (unit-scale normalization);
7. computes sparse Vietoris-Rips persistence in H0, H1, and H2;
8. compares pooled regional landscapes and per-recording landscape norms.

The final cell makes one figure per embedding dimension. The left subfigure shows pooled regional landscapes; the right subfigure shows each recording's real scalar norm against its 100-shuffle null interval.

> **Runtime:** the full analysis is large. Results are checkpointed after every PH computation, so interrupted runs can be resumed. Start with `SMOKE_TEST = True`, then set it to `False` for the complete analysis.

In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import pickle
import random
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from persim import PersLandscapeApprox
from ripser import ripser
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

try:
    from cebra import CEBRA
except ImportError as exc:
    raise ImportError(
        "CEBRA is required. Run this notebook in the repository's topology environment."
    ) from exc

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tda_utils import TDADataManager

sns.set_theme(style="ticks", context="notebook")
plt.rcParams["figure.dpi"] = 140

## Configuration

`N_POINTS` is deliberately global: every point cloud uses exactly this many points. The same time indices are reused across methods and all null realizations for a recording. `K_NEIGHBORS` is smaller than the legacy notebook's 300-neighbor graph to keep 100 H2 null computations per recording tractable; increase it only after a sensitivity check.

The scalar displayed on the right is the combined L2 norm across the homology groups in `SCALAR_HOMOLOGIES`. Set this to `(1,)` for an H1-only comparison.

In [3]:
SEED = 42
REGIONS = ("LGN", "V1")
STIMULI = (
    "color_exchange",
    "luminance_flash",
    "drifting_gratings",
    "chromatic_gratings",
)
EMBED_DIMS = (3,)
METHODS = ("CEBRA", "PCA")

NULL_TYPE = "circular_shift"
N_SHUFFLES = 100
N_POINTS = 1_000
K_NEIGHBORS = 50
MAX_HOMOLOGY = 2
LANDSCAPE_STEPS = 500
LANDSCAPE_DEPTH = 3
SCALAR_HOMOLOGIES = (0, 1, 2)

CEBRA_MAX_ITERATIONS = 30_000
CEBRA_BATCH_SIZE = 512
CEBRA_TIME_OFFSETS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Full run by default. Set True to validate the workflow on one condition,
# one dimension, and two null draws before committing substantial compute.
SMOKE_TEST = False
SMOKE_REGION = "LGN"
SMOKE_STIMULUS = "color_exchange"
SMOKE_RECORDING_LIMIT = 1

CACHE_TAG = (
    f"cshift_v1_n{N_POINTS}_k{K_NEIGHBORS}_"
    f"s{N_SHUFFLES}_cebra{CEBRA_MAX_ITERATIONS}"
)
CACHE_ROOT = REPO_ROOT / "data" / "analysis_results" / "matched_null_cache" / CACHE_TAG
FIGURE_ROOT = REPO_ROOT / "data" / "analysis_results" / "matched_null_figures"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Cache: {CACHE_ROOT}")

Device: cpu
Cache: /Users/emilyekstrum/repos/TDA/data/analysis_results/matched_null_cache/cshift_v1_n1000_k50_s100_cebra30000


## Discover real recordings

Only the unshuffled cleaned files are loaded. Nulls are regenerated here with a documented circular-shift procedure. A length mismatch in a source file is reported and the same `zip(recordings, datas)` behavior used by the original notebooks is retained.

In [4]:
data_manager = TDADataManager()
clean_dir = data_manager.data_paths["clean_spike_data_dir"]


def parse_dataset_name(path: Path) -> tuple[str, str]:
    stem = path.stem
    region, stimulus = stem.split("_", 1)
    return region, stimulus


dataset_paths = {}
for path in sorted(clean_dir.glob("*.pkl")):
    if "shuffled" in path.stem:
        continue
    region, stimulus = parse_dataset_name(path)
    if region in REGIONS and stimulus in STIMULI:
        dataset_paths[(region, stimulus)] = path

manifest_rows = []
for (region, stimulus), path in sorted(dataset_paths.items()):
    with path.open("rb") as handle:
        payload = pickle.load(handle)
    recordings = list(payload["recordings"])
    datas = list(payload["datas"])
    if len(recordings) != len(datas):
        warnings.warn(
            f"{path.name}: {len(recordings)} labels but {len(datas)} matrices; "
            "only paired entries will be analyzed."
        )
    for recording, matrix in zip(recordings, datas):
        manifest_rows.append(
            {
                "region": region,
                "stimulus": stimulus,
                "recording": str(recording),
                "time_points": int(matrix.shape[0]),
                "neurons": int(matrix.shape[1]),
                "path": path,
            }
        )
    del payload, datas
    gc.collect()

manifest = pd.DataFrame(manifest_rows)
if SMOKE_TEST:
    manifest = manifest[
        (manifest.region == SMOKE_REGION)
        & (manifest.stimulus == SMOKE_STIMULUS)
    ].groupby(["region", "stimulus"], as_index=False).head(SMOKE_RECORDING_LIMIT)

display(manifest.drop(columns="path"))

Spike data already available.
All persistence diagrams already available.


/var/folders/st/2v00q15s7bb4yr2r5ghl_42r0000gn/T/ipykernel_72865/2038737969.py:26: UserWarning: V1_drifting_gratings.pkl: 6 labels but 5 matrices; only paired entries will be analyzed.
  warnings.warn(


,region,stimulus,recording,time_points,neurons
0,LGN,chromatic_gratings,C155,136000,15
1,LGN,chromatic_gratings,C159,136000,177
2,LGN,chromatic_gratings,C161,136000,95
3,LGN,color_exchange,d4,34000,44
4,LGN,color_exchange,d5,34000,90
5,LGN,color_exchange,d6,34000,90
6,LGN,color_exchange,C153,34000,150
7,LGN,color_exchange,C155,34000,15
8,LGN,color_exchange,C159,34000,150
9,LGN,color_exchange,C160,34000,55


## Embedding and null helpers

The circular shift is applied to the complete concatenated recording. Every neuron receives an independently sampled nonzero shift. Reducers are fitted only on the real standardized data; null data are transformed through those fixed maps, avoiding 100 refits and keeping the null focused on temporal alignment.

In [5]:
def stable_seed(*parts) -> int:
    token = "|".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(token).digest()[:4], "little")


def as_numpy(matrix) -> np.ndarray:
    if torch.is_tensor(matrix):
        matrix = matrix.detach().cpu().numpy()
    return np.asarray(matrix, dtype=np.float32)


def circular_shift_neurons(X: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    '''Independently circularly shift each neuron's complete time series.'''
    n_time, n_neurons = X.shape
    shifts = rng.integers(1, n_time, size=n_neurons)
    shifted = np.empty_like(X)
    for neuron, offset in enumerate(shifts):
        shifted[:, neuron] = np.roll(X[:, neuron], int(offset))
    return shifted


def matched_indices(n_time: int, key: tuple) -> np.ndarray:
    if n_time < N_POINTS:
        raise ValueError(f"Recording has {n_time} points, fewer than N_POINTS={N_POINTS}.")
    rng = np.random.default_rng(stable_seed(SEED, "indices", *key))
    return np.sort(rng.choice(n_time, size=N_POINTS, replace=False))


def unit_rms_point_cloud(Y: np.ndarray) -> np.ndarray:
    '''Center a point cloud and divide by its root-mean-square radius.'''
    Y = np.asarray(Y, dtype=np.float32)
    Y = Y - Y.mean(axis=0, keepdims=True)
    radius = float(np.sqrt(np.mean(np.sum(Y * Y, axis=1))))
    if not np.isfinite(radius) or radius <= np.finfo(np.float32).eps:
        raise ValueError("Cannot scale-normalize a degenerate point cloud.")
    return Y / radius


def make_cebra(dim: int, seed: int):
    torch.manual_seed(seed)
    return CEBRA(
        model_architecture="offset10-model",
        batch_size=CEBRA_BATCH_SIZE,
        learning_rate=3e-4,
        temperature=1,
        output_dimension=dim,
        max_iterations=CEBRA_MAX_ITERATIONS,
        num_hidden_units=128,
        distance="cosine",
        conditional="time",
        device=DEVICE,
        verbose=False,
        time_offsets=CEBRA_TIME_OFFSETS,
        optimizer="adam",
    )


def fit_reducers(X_real: np.ndarray, dim: int, seed: int):
    scaler = StandardScaler().fit(X_real)
    X_scaled = scaler.transform(X_real).astype(np.float32, copy=False)

    pca = PCA(n_components=dim, svd_solver="randomized", random_state=seed)
    pca.fit(X_scaled)

    cebra_model = make_cebra(dim, seed)
    cebra_model.fit(X_scaled)
    return scaler, {"PCA": pca, "CEBRA": cebra_model}


def transform_cloud(
    X: np.ndarray,
    scaler: StandardScaler,
    reducer,
    sample_idx: np.ndarray,
) -> np.ndarray:
    X_scaled = scaler.transform(X).astype(np.float32, copy=False)
    embedding = reducer.transform(X_scaled)
    return unit_rms_point_cloud(np.asarray(embedding)[sample_idx])

## Persistent homology and checkpointing

PH uses an undirected union-symmetrized Euclidean kNN graph. The filtration threshold is unrestricted; all finite sparse-graph edges can enter. Every cache record contains all three diagrams and complete provenance.

In [6]:
def sparse_knn_distance(point_cloud: np.ndarray) -> csr_matrix:
    n = point_cloud.shape[0]
    k = min(K_NEIGHBORS, n - 1)
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean", n_jobs=-1)
    distances, neighbors = nn.fit(point_cloud).kneighbors(point_cloud)

    rows = np.repeat(np.arange(n), k)
    cols = neighbors[:, 1 : k + 1].reshape(-1)
    vals = distances[:, 1 : k + 1].astype(np.float32).reshape(-1)
    graph = csr_matrix((vals, (rows, cols)), shape=(n, n))
    graph = graph.maximum(graph.T)
    graph.setdiag(0)
    graph.eliminate_zeros()
    return graph


def persistence_diagrams(point_cloud: np.ndarray) -> list[np.ndarray]:
    graph = sparse_knn_distance(point_cloud)
    return ripser(
        graph,
        distance_matrix=True,
        maxdim=MAX_HOMOLOGY,
        thresh=np.inf,
        coeff=2,
    )["dgms"]


def result_path(meta: dict) -> Path:
    state = "real" if meta["null_index"] is None else f"null_{meta['null_index']:03d}"
    name = "__".join(
        [
            meta["region"],
            meta["stimulus"],
            meta["recording"],
            f"d{meta['dimension']}",
            meta["method"].lower(),
            state,
        ]
    )
    return CACHE_ROOT / f"{name}.pkl"


def save_result(path: Path, payload: dict) -> None:
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
    temporary.replace(path)


def expected_paths(base_meta: dict, n_shuffles: int) -> list[Path]:
    paths = []
    for method in METHODS:
        meta = {**base_meta, "method": method, "null_index": None}
        paths.append(result_path(meta))
        for null_index in range(n_shuffles):
            meta = {**base_meta, "method": method, "null_index": null_index}
            paths.append(result_path(meta))
    return paths

## Run the matched analysis

This cell is resumable. A recording/dimension is skipped only when every expected real and null result is already present. PCA dimensions larger than the number of recorded neurons are skipped for both methods so that PCA and CEBRA remain paired.

In [7]:
active_dims = (3,) if SMOKE_TEST else EMBED_DIMS
active_shuffles = 2 if SMOKE_TEST else N_SHUFFLES

for dataset_key, dataset_manifest in manifest.groupby(["region", "stimulus"], sort=True):
    region, stimulus = dataset_key
    path = dataset_manifest.iloc[0]["path"]
    with Path(path).open("rb") as handle:
        payload = pickle.load(handle)
    matrix_by_recording = {
        str(recording): as_numpy(matrix)
        for recording, matrix in zip(payload["recordings"], payload["datas"])
    }
    del payload

    for row in dataset_manifest.itertuples(index=False):
        recording = row.recording
        X_real = matrix_by_recording[recording]
        key = (region, stimulus, recording)
        sample_idx = matched_indices(X_real.shape[0], key)

        for dim in active_dims:
            if dim > X_real.shape[1]:
                print(
                    f"Skip {region}/{stimulus}/{recording}/{dim}d: "
                    f"PCA has only {X_real.shape[1]} neural features."
                )
                continue

            base_meta = {
                "region": region,
                "stimulus": stimulus,
                "recording": recording,
                "dimension": dim,
                "n_points": N_POINTS,
                "k_neighbors": K_NEIGHBORS,
                "null_type": NULL_TYPE,
            }
            if all(path.exists() for path in expected_paths(base_meta, active_shuffles)):
                print(f"Cached: {region}/{stimulus}/{recording}/{dim}d")
                continue

            fit_seed = stable_seed(SEED, "fit", region, stimulus, recording, dim)
            scaler, reducers = fit_reducers(X_real, dim, fit_seed)

            for method, reducer in reducers.items():
                meta = {**base_meta, "method": method, "null_index": None}
                path_out = result_path(meta)
                if not path_out.exists():
                    cloud = transform_cloud(X_real, scaler, reducer, sample_idx)
                    save_result(path_out, {"meta": meta, "dgms": persistence_diagrams(cloud)})

            for null_index in range(active_shuffles):
                null_seed = stable_seed(
                    SEED, "circular_shift", region, stimulus, recording, null_index
                )
                rng = np.random.default_rng(null_seed)
                X_null = circular_shift_neurons(X_real, rng)
                for method, reducer in reducers.items():
                    meta = {**base_meta, "method": method, "null_index": null_index}
                    path_out = result_path(meta)
                    if path_out.exists():
                        continue
                    cloud = transform_cloud(X_null, scaler, reducer, sample_idx)
                    save_result(path_out, {"meta": meta, "dgms": persistence_diagrams(cloud)})
                del X_null
                gc.collect()

            del scaler, reducers
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    del matrix_by_recording
    gc.collect()

KeyboardInterrupt: 

## Convert diagrams to comparable landscapes

Within each embedding dimension, every landscape uses a common filtration grid from zero to the largest finite death value. Infinite H0 classes are removed. The per-homology L2 norm is integrated over this common grid; the right-panel scalar is the Euclidean combination of the selected homology norms.

In [ ]:
def finite_diagrams(dgms) -> list[np.ndarray]:
    cleaned = []
    for homology in range(MAX_HOMOLOGY + 1):
        dgm = np.asarray(dgms[homology], dtype=float)
        if dgm.size == 0:
            cleaned.append(np.empty((0, 2), dtype=float))
            continue
        keep = np.isfinite(dgm).all(axis=1) & (dgm[:, 1] > dgm[:, 0])
        cleaned.append(dgm[keep])
    return cleaned


def landscape_values(dgms, homology: int, stop: float) -> np.ndarray:
    if homology >= len(dgms) or len(dgms[homology]) == 0:
        return np.zeros((1, LANDSCAPE_STEPS), dtype=float)
    landscape = PersLandscapeApprox(
        dgms=dgms,
        hom_deg=homology,
        start=0.0,
        stop=stop,
        num_steps=LANDSCAPE_STEPS,
    )
    values = np.asarray(landscape.values, dtype=float)
    return values[None, :] if values.ndim == 1 else values


def current_result_files(dim: int) -> list[Path]:
    return sorted(CACHE_ROOT.glob(f"*__d{dim}__*.pkl"))


def summarize_dimension(dim: int):
    files = current_result_files(dim)
    if not files:
        raise FileNotFoundError(f"No cached results found for {dim}d.")

    stop = 0.0
    for path in files:
        with path.open("rb") as handle:
            result = pickle.load(handle)
        for dgm in finite_diagrams(result["dgms"]):
            if len(dgm):
                stop = max(stop, float(np.max(dgm[:, 1])))
    if stop <= 0:
        raise ValueError(f"No finite persistence intervals found for {dim}d.")

    grid = np.linspace(0.0, stop, LANDSCAPE_STEPS)
    pooled_sum = defaultdict(lambda: np.zeros((LANDSCAPE_DEPTH, LANDSCAPE_STEPS)))
    pooled_count = defaultdict(int)
    norm_rows = []

    for path in files:
        with path.open("rb") as handle:
            result = pickle.load(handle)
        meta = result["meta"]
        dgms = finite_diagrams(result["dgms"])
        state = "real" if meta["null_index"] is None else "circular shift"
        homology_norms = {}

        for homology in range(MAX_HOMOLOGY + 1):
            values = landscape_values(dgms, homology, stop)
            squared_norm = float(np.sum(np.trapz(values * values, grid, axis=1)))
            homology_norms[homology] = np.sqrt(max(squared_norm, 0.0))

            padded = np.zeros((LANDSCAPE_DEPTH, LANDSCAPE_STEPS))
            depth = min(LANDSCAPE_DEPTH, values.shape[0])
            padded[:depth] = values[:depth]
            pool_key = (meta["method"], meta["region"], state, homology)
            pooled_sum[pool_key] += padded
            pooled_count[pool_key] += 1

        scalar = float(
            np.sqrt(sum(homology_norms[h] ** 2 for h in SCALAR_HOMOLOGIES))
        )
        norm_rows.append(
            {
                **meta,
                "state": state,
                "landscape_norm": scalar,
                **{f"H{h}_norm": homology_norms[h] for h in homology_norms},
            }
        )

    pooled = {key: value / pooled_count[key] for key, value in pooled_sum.items()}
    return pd.DataFrame(norm_rows), pooled, grid


def recording_null_summary(norms: pd.DataFrame) -> pd.DataFrame:
    group_cols = ["dimension", "method", "region", "stimulus", "recording"]
    rows = []
    for key, group in norms.groupby(group_cols, sort=True):
        real = group.loc[group.state == "real", "landscape_norm"]
        null = group.loc[group.state == "circular shift", "landscape_norm"]
        if len(real) != 1 or null.empty:
            continue
        real_value = float(real.iloc[0])
        null_values = null.to_numpy(float)
        rows.append(
            {
                **dict(zip(group_cols, key)),
                "real_norm": real_value,
                "null_mean": float(np.mean(null_values)),
                "null_low": float(np.quantile(null_values, 0.025)),
                "null_high": float(np.quantile(null_values, 0.975)),
                "null_sd": float(np.std(null_values, ddof=1)),
                "z_score": float(
                    (real_value - np.mean(null_values)) / np.std(null_values, ddof=1)
                )
                if np.std(null_values, ddof=1) > 0
                else np.nan,
                "p_upper": float((1 + np.sum(null_values >= real_value)) / (len(null_values) + 1)),
                "n_null": len(null_values),
            }
        )
    return pd.DataFrame(rows)

## Figure: pooled landscapes and per-recording norms

The left panel pools recordings and stimuli with equal weight within region. Solid lines are real data and dashed lines are the circular-shift mean; the first three landscape layers are shown. On the right, each horizontal offset corresponds to one recording: the vertical line is its 95% null interval, the open marker is its null mean, and the filled marker is its real value.

In [ ]:
REGION_COLORS = {"LGN": "#0072B2", "V1": "#D55E00"}
STIMULUS_LABELS = {
    "color_exchange": "Color\nexchange",
    "luminance_flash": "Luminance\nflash",
    "drifting_gratings": "Drifting\ngratings",
    "chromatic_gratings": "Chromatic\ngratings",
}


def plot_dimension(dim: int, norms: pd.DataFrame, pooled: dict, grid: np.ndarray):
    paired = recording_null_summary(norms)
    fig = plt.figure(figsize=(23, 10), constrained_layout=True)
    left, right = fig.subfigures(1, 2, width_ratios=(1.18, 1.0), wspace=0.04)

    left.suptitle("Pooled persistence landscapes by region", fontweight="bold")
    left_axes = left.subplots(len(METHODS), MAX_HOMOLOGY + 1, squeeze=False)
    depth_alpha = np.linspace(1.0, 0.45, LANDSCAPE_DEPTH)
    for row, method in enumerate(METHODS):
        for homology in range(MAX_HOMOLOGY + 1):
            ax = left_axes[row, homology]
            for region in REGIONS:
                color = REGION_COLORS[region]
                for state, linestyle in (("real", "-"), ("circular shift", "--")):
                    values = pooled.get((method, region, state, homology))
                    if values is None:
                        continue
                    for depth in range(LANDSCAPE_DEPTH):
                        label = f"{region}, {state}" if depth == 0 else None
                        ax.plot(
                            grid,
                            values[depth],
                            color=color,
                            linestyle=linestyle,
                            alpha=depth_alpha[depth],
                            linewidth=2.2 if depth == 0 else 1.4,
                            label=label,
                        )
            ax.set_title(f"H{homology}")
            if homology == 0:
                ax.set_ylabel(f"{method}\nLandscape value")
            if row == len(METHODS) - 1:
                ax.set_xlabel("Filtration value")
            if row == 0 and homology == MAX_HOMOLOGY:
                ax.legend(frameon=False, fontsize=8)

    right.suptitle("Landscape norm per recording", fontweight="bold")
    right_axes = right.subplots(len(METHODS), len(REGIONS), squeeze=False)
    for row, method in enumerate(METHODS):
        for col, region in enumerate(REGIONS):
            ax = right_axes[row, col]
            subset = paired[(paired.method == method) & (paired.region == region)]
            for stim_index, stimulus in enumerate(STIMULI):
                stim_data = subset[subset.stimulus == stimulus].sort_values("recording")
                if stim_data.empty:
                    continue
                offsets = np.linspace(-0.22, 0.22, len(stim_data)) if len(stim_data) > 1 else [0.0]
                for offset, record in zip(offsets, stim_data.itertuples(index=False)):
                    x = stim_index + offset
                    ax.vlines(x, record.null_low, record.null_high, color="0.55", linewidth=2)
                    ax.scatter(
                        x,
                        record.null_mean,
                        s=34,
                        facecolors="white",
                        edgecolors="0.35",
                        zorder=3,
                    )
                    ax.scatter(
                        x,
                        record.real_norm,
                        s=42,
                        color=REGION_COLORS[region],
                        edgecolors="white",
                        linewidth=0.5,
                        zorder=4,
                    )
                    ax.plot(
                        [x, x],
                        [record.null_mean, record.real_norm],
                        color=REGION_COLORS[region],
                        alpha=0.35,
                        linewidth=0.8,
                    )
            ax.set_xticks(range(len(STIMULI)), [STIMULUS_LABELS[s] for s in STIMULI])
            ax.tick_params(axis="x", labelsize=8)
            ax.set_title(region)
            if col == 0:
                ax.set_ylabel(f"{method}\nCombined L2 norm")
            ax.grid(axis="y", color="0.9", linewidth=0.7)

    fig.suptitle(
        f"Matched circular-shift null: CEBRA versus PCA ({dim} dimensions)",
        fontsize=18,
        fontweight="bold",
    )
    return fig, paired


all_norms = []
all_paired = []
for dim in active_dims:
    norms, pooled, grid = summarize_dimension(dim)
    fig, paired = plot_dimension(dim, norms, pooled, grid)
    png_path = FIGURE_ROOT / f"matched_null_cebra_vs_pca_{dim}d.png"
    pdf_path = FIGURE_ROOT / f"matched_null_cebra_vs_pca_{dim}d.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    all_norms.append(norms)
    all_paired.append(paired)

norms_df = pd.concat(all_norms, ignore_index=True)
paired_summary_df = pd.concat(all_paired, ignore_index=True)
norms_df.to_csv(FIGURE_ROOT / "landscape_norms_all_null_draws.csv", index=False)
paired_summary_df.to_csv(FIGURE_ROOT / "landscape_norms_recording_summary.csv", index=False)

display(paired_summary_df.head())
print(f"Figures and tables saved to {FIGURE_ROOT}")

## Interpretation notes

- A recording is the independent observational unit. The 100 shuffles estimate its null distribution; they are not 100 biological replicates.
- `p_upper` is the finite-sample upper-tail empirical p-value `(1 + # null >= real) / 101`.
- CEBRA and PCA use the same standardized neural matrix, sampled time indices, point count, scale normalization, kNN graph, filtration, and landscape grid.
- Reducers are fitted on real data and held fixed for null transforms. This tests whether coordinated temporal structure is needed to occupy the observed geometry under each learned representation.
- Circular shifting the complete concatenated recording preserves each neuron's values and autocorrelation but can cross inferred trial boundaries because explicit trial boundaries are absent from the repository data.
- Dimensions exceeding a recording's neuron count are excluded from both methods to keep comparisons paired.